#Description 描述
---
資料集來源(https://www.kaggle.com/datasets/kelvinobiri/credit-card-transactions)
---
#雖然資料集叫credit card但整體比較像金融活動的資料集

| 英文            | 解釋                                           |                     |
|-----------------|------------------------------------------------|---------------------|
| step            | Time step of the transaction                   | 交易的時間跨度      |
| type            | Type of transaction (e.g., TRANSFER, CASH_OUT) | 交易類型（例如TRANSFER，CASH_OUT）。 |
| amount          | Amount involved in the transaction             | 交易涉及的金額      |
| nameOrig        | ID of sender account                           | 發件人帳戶的ID      |
| oldbalanceOrg   | Sender’s balance before the transaction        | 交易前發件人的餘額  |
| newbalanceOrig  | Sender’s balance after the transaction         | 交易後發件人的餘額  |
| nameDest        | ID of receiver account                         | 接收器帳戶的ID      |
| oldbalanceDest  | Receiver’s balance before the transaction      | 交易前接收者的餘額  |
| newbalanceDest  | Receiver’s balance after the transaction       | 交易後接收者的餘額  |
| isFraud         | Target variable: 1 if fraudulent, 0 otherwise  | 目標變量：1是欺詐，0否 |
| isPayment       | Indicates if the transaction is a payment      | 指示交易是否是付款  |
| isMovement      | Indicates if it involved a balance change      | 指示是否涉及餘額變化 |
| accountDiff     | Difference in account balances (derived feature) | 帳戶餘額的差異（派生功能） |



可以看到詐騙的資料量和整體的資料量比例是很懸殊的，這要列入模型選擇的考量

In [1]:
import pandas as pd

# 讀取資料
df = pd.read_csv("transactions.csv")  

# 顯示前5筆資料
print(df.head())

# 顯示欄位資訊與缺失值檢查
print(df.info())
print(df.isnull().sum())

# 統計詐騙比例
fraud_rate = df['isFraud'].mean()
print(f"詐騙比例：{fraud_rate:.4f}")


   step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     8  CASH_OUT  158007.12   C424875646           0.00            0.00   
1   236  CASH_OUT  457948.30  C1342616552           0.00            0.00   
2    37   CASH_IN  153602.99   C900876541    11160428.67     11314031.67   
3   331  CASH_OUT   49555.14   C177696810       10865.00            0.00   
4   250  CASH_OUT   29648.02   C788941490           0.00            0.00   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  
0  C1298177219       474016.32      1618631.97        0  
1  C1323169990      2720411.37      3178359.67        0  
2   C608741097      3274930.56      3121327.56        0  
3   C462716348            0.00        49555.14        0  
4  C1971700992        56933.09        86581.10        0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199999 entries, 0 to 199998
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   

--------------------------------------------------------------------

從上面的欄位描述可以看到nameOrig 跟nameDest 是名稱對於資料分析沒有幫助所以刪掉。

In [2]:
# 移除帳號 ID 欄位
df = df.drop(['nameOrig', 'nameDest'], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199999 entries, 0 to 199998
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            199999 non-null  int64  
 1   type            199999 non-null  object 
 2   amount          199999 non-null  float64
 3   oldbalanceOrg   199999 non-null  float64
 4   newbalanceOrig  199999 non-null  float64
 5   oldbalanceDest  199999 non-null  float64
 6   newbalanceDest  199999 non-null  float64
 7   isFraud         199999 non-null  int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 12.2+ MB


--------------------------------------------------------------------

處理不是數值的欄位，type欄位是object使用one hot encoding處理

In [3]:
# 顯示 type 欄位有哪些類別
print(df['type'].unique())

# 使用 One-hot encoding 處理 'type' 欄位
df = pd.get_dummies(df, columns=['type'], drop_first=True)

# 檢查轉換後的欄位名稱
print(df.columns)

['CASH_OUT' 'CASH_IN' 'PAYMENT' 'TRANSFER' 'DEBIT']
Index(['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
       'newbalanceDest', 'isFraud', 'type_CASH_OUT', 'type_DEBIT',
       'type_PAYMENT', 'type_TRANSFER'],
      dtype='object')


--------------------------------------------------------------------

選這四個模型處理詐騙預測問題，是因為它們在處理類別不平衡、非線性特徵、可解釋性與效能上各有優勢

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

#分割特徵與目標欄位
X = df.drop('isFraud', axis=1)
y = df['isFraud']

#拆分訓練與測試資料（80% 訓練，20% 測試）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#預測模型
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "XGBoost": XGBClassifier( eval_metric='logloss', scale_pos_weight=300),  
    "LightGBM": LGBMClassifier(class_weight='balanced'),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
}

for name, model in models.items():
    print(f"\n=== {name} ===")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    print("ROC AUC Score:", roc_auc_score(y_test, y_prob))



=== Logistic Regression ===
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96     39944
           1       0.02      0.98      0.04        56

    accuracy                           0.92     40000
   macro avg       0.51      0.95      0.50     40000
weighted avg       1.00      0.92      0.96     40000

ROC AUC Score: 0.9919159144230495

=== XGBoost ===
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     39944
           1       0.83      0.93      0.87        56

    accuracy                           1.00     40000
   macro avg       0.91      0.96      0.94     40000
weighted avg       1.00      1.00      1.00     40000

ROC AUC Score: 0.9996669444364967

=== LightGBM ===
[LightGBM] [Info] Number of positive: 226, number of negative: 159773
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001052 sec